Modelo de aprendizaje automático - Para saber si un estudiante Aprobará o No aprobará la carrera - Algoritmo Naive Bayes

In [ ]:
#IMPORTAR LAS LIBRERÍAS
# Librería para trabajar con los comandos del Sistema Operativo
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine 


#Calcular valores numéricos - Vectores - Matrices
import numpy as np
#Manipulación de los datos - Cargar - Limpiar - Transformación
import pandas as pd
# Visualizaciones de los datos. Gráficos: Torta - Área - linea - Dispersión
import matplotlib.pyplot as plt
# Dividir los datos en entrenamiento/prueba - Realizar las validaciones cruzadas
from sklearn.model_selection import train_test_split, cross_val_score
# Permite encadenar pasos secuencias - encadenados
from sklearn.pipeline import Pipeline
# Sirve para rellenar valores faltantes o nulos (nan) - Remplazar: Media - Mediana - Valor
from sklearn.impute import SimpleImputer

# Crear el modelo ML Supervisado - Categorías
from sklearn.naive_bayes import GaussianNB

# Precisión de aciertos del modelo - Matrix de confusión - Reporte de clasificación de aprendizaje del modelo
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

np.random.seed(42)

print('Librerías cargadas de forma correcta')


1.- DATOS

In [ ]:
# Leer los datos históricos - desde nuestra base de datos
# Invocar al método para leer las variables de entorno
load_dotenv()

# Leyendo las variables de entorno
host = os.getenv('DB_HOST')
puerto = os.getenv('DB_PORT')
usuario = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
base_datos = os.getenv('DB_NAME')

# Crear la conexión a la base de datos
#                            script    ://usuario:password@host:puerto/base_datos

conexion = create_engine(
    f'mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{base_datos}'
)

# Creando nuestras consultas
consulta = 'SELECT * FROM estudiantes'


# Ejecutar la consulta
datos = pd.read_sql(
    consulta , 
    conexion
)

print('Cantidad de registros del Data Frame: ', len(datos) )
datos.head()

# Revisión de los de estructura de los datos
print('Distribucíon de la variable objetivo')
print(datos['aprobo'].value_counts())

# Visualización de cuantos aprobaron y cuanto no - Gráfico
plt.title('Distribución de la variable objetivo')
plt.ylabel('Cantidad de estudiantes')
# kind= 'bar' 'barh' - 'line' - scatter - hist - area - pie
datos['aprobo'].value_counts().sort_index().plot(kind='bar' , color=['green','red'])
plt.xticks([0,1], ['No aprobado', 'Aprobado'],rotation=0)
# plt.legend()
plt.grid()
plt.show()


2.- SELECCIÓN DE VARIABLES

In [ ]:
# X= Variables que el modelo utilizará para aprender
# y = Variable que queremos predecir
# datos.head()
# edad
# horas_estudio_semana
# asistencia_pct
# tareas_entregadas_pct
# nota_parcial
# participacion_clase
# horas_sueno
# trabaja
variables = [
    'edad',
    'horas_estudio_semana',
    'asistencia_pct',
    'tareas_entregadas_pct',
    'nota_parcial',
    'participacion_clase',
    'horas_sueno',
    'trabaja'
]
# Configuración de PAndas para poder visualizar las filas o columnas (TODAS)
pd.set_option('display.max_rows',None)
# pd.set_option('display.max_cols',None)
# Entrenamiento
X= datos[variables]
# Pruebas
y= datos["aprobo"]
print('Variables predictoras')
print(X.columns.tolist())
print('Variable objetivo')
# Nombre de la columna
print(y.name)
# Tipo de datos
print(y.dtype)
# Valores de la columna
print(y.values)
# Cantidad
# print(y.count)
# ===============================================================================
# Debo verificar la cantidad de registros y la cantidad de variables predictoras
# ===============================================================================
print('Cantidad de registros y variables predictoras')
print(X.shape)


3.- PREPARACIÓN DE LOS DATOS

In [ ]:
# Hacer una copia de los datos X = Entrenamiento
X = X.copy()
# print(X)
#Comprobando si hay o no datos faltantes
print(X.isnull().sum()) 


4.- TRAINT / TEST
Dividir los datos en dos grupos

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size= 0.20,
    random_state= 42,
    stratify= y
)
print('Datos de entrenamiento: ', X_train.shape)
print('Datos de prueba: ', X_test.shape)

5.- ENTRENAMIENTO

In [ ]:
# Trabajando con Pipeline - Lograr ejecutar multiples opciones de forma secuencia
modelo = Pipeline(
    [
        # Completa valores faltantes
        ('imputer',SimpleImputer(strategy='median')),

        # Indicar el modelo a utilizar - Naive Bayes
        ('clasificación', GaussianNB( ))
    ]
)
# Entrenamiento del modelo
modelo.fit(X_train,y_train)
print('Modelo entrenado de forma correcta')


6.- VALIDACIÓN

In [ ]:
# Validación cruzada cv-5
# 1.- Dividir Entramiento 5 partes
# 2.- Se entrena con 4 partes
# 3.- Se valida con la parte restante
# 4.- Se repita cinco veces
# 5.- Se calcula el promedio
# OJO: Nosotros no tocamos la parte de prueba
resultados = cross_val_score(
    modelo,
    X_train,
    y_train,
    cv = 5,
    scoring = 'accuracy'
)
print('Precisión de cada validación: ', np.round(resultados,3))

print('Precisión promedio: ', np.round(resultados.mean(),3))

print('Desviación standar: ', np.round(resultados.std(),3))


7.- EVALUACIÓN

In [ ]:
# Calcular la exactitud - Precisión
y_prediccion = modelo.predict(X_test)
print(y_prediccion)
precision = accuracy_score(y_test,y_prediccion)
print('Precisión de prueba: ', round(precision,3))

# Matriz de confusión
matrizConfusion = confusion_matrix(y_test,y_prediccion)
print('Matriz de confusión')
print(matrizConfusion)

# Generar el reporte de clasificación
print('Reporte de clasificación')
print(classification_report(
    y_test,
    y_prediccion,
    target_names=['No aprobó','Aprobo']
))


Visualización de la matriz de confusión

In [ ]:
# Matriz de confusión
ConfusionMatrixDisplay.from_predictions(
y_test,
y_prediccion,
display_labels=["No aprobó", "Aprobó"],
cmap="Reds",
values_format="d"
)

plt.title("Matriz de confusión - Algoritmo Naive Bayes")
plt.xlabel("Valor predicho")
plt.ylabel("Valor real")
plt.show()

8.- DIAGNOSTICO

In [ ]:
# Investigar: Sobreajuste (overfitting) - Subajuste (underfitting) - Buena generalización
precision_entrenamiento = accuracy_score(y_train,modelo.predict(X_train))

precision_prueba = accuracy_score(y_test,modelo.predict(X_test)) 

print('Diagnostrico')
print('Precisión de entrenamiento: ', round(precision_entrenamiento,3))
print('Precisión de prueba: ', round(precision_prueba,3))
print('Diferencia: ', round(precision_entrenamiento,3) - round(precision_prueba,3) )


9.- USAR EL MODELO - Probarlo

In [ ]:
# Probando el modelo - Simulando carga de datos

nuevo_estudiante = pd.DataFrame({
        'edad' : [59],
        'horas_estudio_semana': [2],
        'asistencia_pct': [10],
        'tareas_entregadas_pct': [70],
        'nota_parcial': [8],
        'participacion_clase': [9],
        'horas_sueno': [5.0],
        'trabaja': [1]
})
# ===========================================================
# Muestro el dataset actual, para ver las columnas que tengo
# ===========================================================
datos.head()

prediccion = modelo.predict(nuevo_estudiante)
# *************************************************
# Verificación de Probabilidades encontradas
# *************************************************
print('************************************************************')
probabilidades = modelo.predict_proba(nuevo_estudiante)
print('Determinación de las probabilidades - Naive Bayes')

# print('Probabilidad de no aprobar: ', round(probabilidades[0][0],3))
# print('Probabilidad de aprobar: ', round(probabilidades[0][1],3))

# ---------------------------------------------------------------------
print('Probabilidades: ', probabilidades[0])
print(f'Probabilidad de no aprobar: {probabilidades[0][0]:.2f}')
print(f'Probabilidad de aprobar: {probabilidades[0][1]:.2f}')
print('Probabilidad de aprobar: ', round(probabilidades[0][1],3))
print('Representación del 100%')
print(f'Probabilidad de aprobar: {probabilidades[0][1] * 100:.0f}%')
print('************************************************************')




# *************************************************
print('Predicción: ', prediccion[0])
if prediccion[0] == 1:
    print('El estudiante aprobo')
else:
    print('El estudiante no aprobo')

# If ternario - doble 
# let valor1 = 10
# let valor2 = 20
#               Condición            Respuesta 1     Respuesta 2
# resultado = (valor1 > valor2)? valor1 + valor2 : valor1*valor2
print('Predicción: ', 'El estudiante aprobo' if prediccion[0] == 1 else 'El estudiante no aprobo')


